# SILVER (Dados Limpos) 
Dados validados e limpos<br>
Tipos de dados corretos<br>
Valores nulos tratados<br>
Duplicatas removidas<br>

## PROCESSAMENTO DOS DADOS

### IMPORTAÇÃO DAS BIBLIOTECAS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime
import seaborn as sns
import numpy as np

### CARREGAMENTO DOS DADOS

In [ ]:
bronze_path = 'data/bronze/dados_brutos.csv'
df = pd.read_csv(bronze_path)

## TRATAMENTO DOS DADOS


### ALTERAÇÃO DOS TIPOS DE DADOS

In [ ]:
# Identificador
df['CODIGO_CLIENTE'] = df['CODIGO_CLIENTE'].astype(str)

# Categóricas 
df['UF'] = df['UF'].astype('category')
df['ESCOLARIDADE'] = df['ESCOLARIDADE'].astype('category')
df['ESTADO_CIVIL'] = df['ESTADO_CIVIL'].astype('category')

# Booleanas 
df['CASA_PROPRIA'] = df['CASA_PROPRIA'].map({'Sim': True, 'Não': False}).astype(np.bool_)
df['OUTRA_RENDA'] = df['OUTRA_RENDA'].map({'Sim': True, 'Não': False}).astype(np.bool_)
df['TRABALHANDO_ATUALMENTE'] = df['TRABALHANDO_ATUALMENTE'].map({'Sim': True, 'Não': False}).astype(np.bool_)

# Numéricas inteiras 
int_cols = ['IDADE', 'QT_FILHOS', 'QT_IMOVEIS', 'TEMPO_ULTIMO_EMPREGO_MESES', 'QT_CARROS', 'SCORE']
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.int64)

# Valores monetários 
float_cols = ['VL_IMOVEIS', 'OUTRA_RENDA_VALOR', 'ULTIMO_SALARIO', 'VALOR_TABELA_CARROS']
for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float64)

### PADRONIZAÇÃO DOS VALORES TEXTUAIS 

In [ ]:
text_cols = [c for c in df.columns if df[c].dtype == 'object']
for c in text_cols:
    df[c] = df[c].astype(str).str.strip().str.upper()

### TRATAMENTO DE NULOS

In [ ]:
df.isnull().sum()

In [ ]:
df.replace('SEM DADOS',np.nan, inplace = True)
df['ULTIMO_SALARIO'] = df['ULTIMO_SALARIO'].fillna((df['ULTIMO_SALARIO'].median()))

### TRATAMENTO DE OUTLIERS


In [ ]:
moda = df['QT_FILHOS'].mode()[0]
df.loc[df['QT_FILHOS'] > 3, 'QT_FILHOS'] = moda

### TRATAMENTO DE DUPLICATAS

In [ ]:
df.duplicated().sum()
df.info()

### CRIAÇÃO DE COLUNA RENDA_TOTAL

In [ ]:
df['RENDA_TOTAL'] = df['ULTIMO_SALARIO'].fillna(0) + df['OUTRA_RENDA_VALOR'].fillna(0)


## PÓS ANÁLISE EXPLORATÓRIA (EDA) 

In [ ]:
num = []
for i in df.columns[0:20].tolist():
        if df.dtypes[i] == 'int64' or df.dtypes[i] == 'float64':            
            print(i, ':' , df.dtypes[i]) 
            num.append(i)

In [ ]:
cat = []
for i in df.columns[0:20].tolist():
        if df.dtypes[i] == 'bool_' or df.dtypes[i] == 'category':            
            print(i, ':' , df.dtypes[i]) 
            cat.append(i)           

In [ ]:
plt.rcParams["figure.figsize"] = [20, 15]
plt.rcParams["figure.autolayout"] = True

f, axes = plt.subplots(3, 4)

axes = axes.flatten()

for i, col in enumerate(num):
    sns.boxplot(data=df, y=col, ax=axes[i])

for j in range(len(num), len(axes)):
    axes[j].set_visible(False)

plt.show()

In [ ]:
plt.rcParams["figure.figsize"] = [20, 15]
plt.rcParams["figure.autolayout"] = True

f, axes = plt.subplots(4, 3) 

linha = 0
coluna = 0

for i in num:
    sns.histplot(data = df, x=i, ax=axes[linha][coluna])    
    coluna += 1
    if coluna == 3:
        linha += 1
        coluna = 0            

plt.show()

In [ ]:

plt.rcParams["figure.figsize"] = [15.00, 22.00]
plt.rcParams["figure.autolayout"] = True


f, axes = plt.subplots(3, 2) 

linha = 0
coluna = 0

for i in cat:    
    sns.countplot(data = df, x=i, ax=axes[linha][coluna])
    
    coluna += 1
    if coluna == 2:
        linha += 1
        coluna = 0            

plt.show()



In [ ]:
plt.rcParams["figure.figsize"] = (18, 8)

corr = df.select_dtypes(include=[np.number]).corr()

ax = sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Matriz de Correlação (somente variáveis numéricas)")
plt.show()

In [ ]:

sns.lmplot(x = "VL_IMOVEIS", y = "SCORE", data = df);

In [ ]:

sns.lmplot(x = "RENDA_TOTAL", y = "SCORE", data = df);

In [ ]:

sns.lmplot(x = "TEMPO_ULTIMO_EMPREGO_MESES", y = "SCORE", data = df);

## SALVAR NA CAMADA SILVER

### ADICIONAR INFORMAÇÃO DE TRATAMENTO DOS DADOS
 

In [ ]:
df['DATA_TRATAMENTO'] = datetime.now()

In [ ]:
silver_path = 'data/silver/dados_limpos.csv'
df.to_csv(silver_path, index=False)
print(f'\nDados limpos salvos: {silver_path} (shape={df.shape})')